In [1]:
# Install project dependencies
!pip install -q shap openai xgboost scikit-learn joblib pandas numpy

In [2]:
import os
from google.colab import userdata
from openai import OpenAI

# Retrieve API key securely from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [3]:
from pathlib import Path

# Define project directory
PROJECT_ROOT = Path("/content/INFS-348-Final-Project")

# Required project files
required_files = [
    "SRC/Agent/prediction_tool.py",
    "SRC/Agent/explanation_tool.py",
    "SRC/Agent/agent.py",
    "Model/fraud_xgboost_model.joblib"
]

# Check that all required files exist
for file in required_files:
    file_path = PROJECT_ROOT / file
    status = "FOUND" if file_path.exists() else "MISSING"
    print(f"{status}: {file}")

MISSING: SRC/Agent/prediction_tool.py
MISSING: SRC/Agent/explanation_tool.py
MISSING: SRC/Agent/agent.py
MISSING: Model/fraud_xgboost_model.joblib


In [ ]:
from google.colab import files
import os
from google.colab import files
import os

# Upload the prediction tool and trained model
uploaded = files.upload()

# Recreate the required project folder structure
os.makedirs("/content/INFS-348-Final-Project/SRC/Agent", exist_ok=True)
os.makedirs("/content/INFS-348-Final-Project/Model", exist_ok=True)

# Place uploaded files in their correct directories
for filename, content in uploaded.items():

    if filename == "prediction_tool.py":
        destination = "/content/INFS-348-Final-Project/SRC/Agent/prediction_tool.py"

    elif filename == "fraud_xgboost_model.joblib":
        destination = "/content/INFS-348-Final-Project/Model/fraud_xgboost_model.joblib"

    else:
        print(f"Skipped unexpected file: {filename}")
        continue

    with open(destination, "wb") as file:
        file.write(content)

    print(f"Uploaded successfully: {filename}")

print("\nProject files are ready for testing!")


In [ ]:
import os

project_root = "/content/INFS-348-Final-Project"

required_files = [
    "SRC/Agent/prediction_tool.py",
    "Model/fraud_xgboost_model.joblib"
]

print("FILE VERIFICATION")
print("-----------------")

for file_path in required_files:
    full_path = os.path.join(project_root, file_path)
    status = "Found" if os.path.exists(full_path) else "Missing"
    print(f"{file_path}: {status}")


In [ ]:
import sys

# Add the agent folder to Python's import path
sys.path.insert(
    0,
    "/content/INFS-348-Final-Project/SRC/Agent"
)

# Import the prediction tool
from prediction_tool import predict_transaction, feature_columns, threshold

print("PREDICTION TOOL IMPORT SUCCESSFUL!")
print("----------------------------------")
print("Model loaded successfully.")
print("Number of expected features:", len(feature_columns))
print("Classification threshold:", threshold)


In [ ]:
import pandas as pd

# Upload the original dataset
from google.colab import files
uploaded = files.upload()

# Load the uploaded CSV
filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()


In [ ]:
# Select the first known fraudulent transaction
sample_transaction = df[df["is_fraud"] == 1].iloc[0]

# Extract the 24 model features
transaction_data = sample_transaction[feature_columns].to_dict()

# Submit transaction to our prediction tool
result = predict_transaction(transaction_data)

# Display results
print("TRANSACTION PREDICTION")
print("----------------------")
print("Actual classification:", "Fraudulent" if sample_transaction["is_fraud"] == 1 else "Legitimate")
print("Model prediction:", result["prediction"])
print("Fraud probability:", result["fraud_probability"])
print("Classification threshold:", result["threshold"])


In [ ]:
# Select the first known legitimate transaction
sample_transaction = df[df["is_fraud"] == 0].iloc[0]

# Extract transaction features
transaction_data = sample_transaction[feature_columns].to_dict()

# Submit transaction to prediction tool
result = predict_transaction(transaction_data)

# Display results
print("TRANSACTION PREDICTION")
print("----------------------")
print("Actual classification: Legitimate")
print("Model prediction:", result["prediction"])
print("Fraud probability:", result["fraud_probability"])
print("Classification threshold:", result["threshold"])


In [ ]:
!pip install shap

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

file_path = "/content/INFS-348-Final-Project/SRC/Agent/explanation_tool.py"

print("File found:", os.path.exists(file_path))

In [ ]:
from explanation_tool import explain_transaction

print("SHAP explanation tool imported successfully!")

In [ ]:
# Select the first known fraudulent transaction
sample_transaction = df[df["is_fraud"] == 1].iloc[0]

# Extract its 24 model features
transaction_data = sample_transaction[feature_columns].to_dict()

# Generate SHAP explanation
explanation = explain_transaction(transaction_data)

# Display the results
print("SHAP EXPLANATION")
print("----------------")

for item in explanation["top_features"]:
    print(
        f'{item["feature"]}: '
        f'{item["shap_value"]} '
        f'({item["impact"]})'
    )

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

file_path = "/content/INFS-348-Final-Project/SRC/Agent/agent.py"

print("File found:", os.path.exists(file_path))

In [ ]:
!pip install -q --upgrade openai

In [ ]:
import os
from google.colab import userdata

# Retrieve the key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("OpenAI API key loaded successfully!")

In [ ]:
!pip install --no-cache-dir --force-reinstall --upgrade openai

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

print("OpenAI client initialized successfully!")

In [ ]:
# Get the existing transaction analysis from our agent
result = analyze_transaction(transaction_data)

# Send a compact summary to the LLM
response = client.responses.create(
    model="gpt-6-luna",
    instructions=(
        "You are a credit card fraud analysis assistant. "
        "Explain the model's prediction and the main SHAP "
        "contributions in 2-3 concise sentences. "
        "Do not override the model's classification. "
        "Mention that predictions are estimates, not proof of fraud."
    ),
    input=str({
        "prediction": result["prediction"],
        "fraud_probability": result["fraud_probability"],
        "threshold": result["classification_threshold"],
        "top_features": result["top_contributing_features"]
    }),
    max_output_tokens=120
)

from IPython.display import display, Markdown

display(Markdown(response.output_text))

In [ ]:
import sys
import importlib

# Add project agent directory to Python's import path
sys.path.insert(
    0,
    "/content/INFS-348-Final-Project/SRC/Agent"
)

# Import the updated agent
import agent
importlib.reload(agent)

from agent import analyze_transaction

print("Agent loaded:", callable(analyze_transaction))

In [ ]:
print("Transaction data loaded:", "transaction_data" in globals())
print("Dataset loaded:", "df" in globals())

In [ ]:
from google.colab import files
import pandas as pd
import os

# Upload the original dataset
uploaded = files.upload()

In [ ]:
# Load the dataset
df = pd.read_csv("credit_card_fraud_2026.csv")

print("Dataset loaded:", df.shape)

In [ ]:
# Select the same first fraudulent transaction
sample_transaction = df[df["is_fraud"] == 1].iloc[0]

# Recreate transaction input
transaction_data = sample_transaction[feature_columns].to_dict()

print("Transaction data restored:", len(transaction_data) == 24)

In [ ]:
from prediction_tool import feature_columns

print("Feature columns loaded:", len(feature_columns))

In [ ]:
sample_transaction = df[df["is_fraud"] == 1].iloc[0]

transaction_data = sample_transaction[feature_columns].to_dict()

print("Transaction data restored:", len(transaction_data) == 24)

In [ ]:
result = analyze_transaction(transaction_data)

print("CREDIT CARD FRAUD ANALYSIS")
print("--------------------------")
print("Prediction:", result["prediction"])
print("Fraud probability:", result["fraud_probability"])
print("Threshold:", result["classification_threshold"])

print("\nTOP CONTRIBUTING FEATURES")
for feature in result["top_contributing_features"]:
    print(
        feature["feature"],
        ":", feature["shap_value"],
        "-", feature["impact"]
    )

print("\nAI AGENT EXPLANATION")
print("\nAI AGENT EXPLANATION")
print("--------------------")
print(result["explanation"].replace(". ", ".\n"))

In [ ]:
sample_legitimate = df[df["is_fraud"] == 0].iloc[0]
legitimate_transaction = sample_legitimate[feature_columns].to_dict()

result_legitimate = analyze_transaction(legitimate_transaction)

print("ACTUAL CLASS: Legitimate")
print("MODEL PREDICTION:", result_legitimate["prediction"])
print("FRAUD PROBABILITY:", result_legitimate["fraud_probability"])

print("\nAI AGENT EXPLANATION")
print("--------------------")
print(result_legitimate["explanation"].replace(". ", ".\n"))